<a href="https://colab.research.google.com/github/gohzhihwee/stuffs/blob/main/Option_Pricing_Workbook_pt3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import scipy.stats as ss
from scipy.stats import lognorm, norm

In [ ]:
def bs_call_mc(S, K, r, sigma, T, t, Ite):
  data = np.zeros((Ite, 2))
  z = np.random.normal(0, 1, [1, Ite])
  ST = S*np.exp((T - t)*(r - 0.5 * sigma**2) + sigma * np.sqrt(T - t) * z)
  data[:, 1] = ST - K
  print(data)

  average = np.sum(np.amax(data, axis=1)) / float(Ite)

  return np.exp(-r * (T - t)) * average

In [ ]:
def bs_put_mc(S, K, r, sigma, T, t, Ite):
  data = np.zeros((Ite, 2))
  z = np.random.normal(0, 1, [1, Ite])
  ST = S*np.exp((T - t)*(r - 0.5 * sigma**2) + sigma * np.sqrt(T - t) * z)
  data[:, 1] = K - ST

  average = np.sum(np.amax(data, axis=1)) / float(Ite)

  return np.exp(-r * (T - t)) * average

In [ ]:
def SDE_vol(v0, kappa, theta, sigma, T, M, Ite, rand, row, cho_matrix):
  dt = T/M
  v = np.zeros((M + 1, Ite), dtype=float)
  v[0] = v0
  sdt = np.sqrt(dt)
  for t in range(1, M + 1):
    ran = np.dot(cho_matrix, rand[:, t])
    v[t] = np.maximum(0, v[t - 1] + kappa * (theta - v[t - 1]) * dt + sigma * sdt * ran[row])
  return v

In [ ]:
def Heston_paths(S0, r, v, row, cho_matrix):
  S = np.zeros((M + 1, Ite), dtype=float)
  S[0] = S0
  for t in range(1, M + 1, 1):
    ran = np.dot(cho_matrix, rand[:, t])
    S[t] = S[t - 1] * np.exp((r - 0.5 * v[t]) * dt + np.sqrt(dt) * np.sqrt(v[t]) * ran[row])
  return S

In [ ]:
def random_number_gen(M, Ite):
  rand = np.random.standard_normal((2, M + 1, Ite))
  return rand

In [ ]:
def heston_call_mc(S, K, r, T, t):
  payoff = np.maximum(0, S[-1, :] - K)
  average = np.mean(payoff)
  return np.exp(-r * (T - t)) * average

In [ ]:
def heston_put_mc(S, K ,r, T, t):
  payoff = np.maximum(0, K - S[-1, :])
  average = np.mean(payoff)
  return np.exp(-r * (T - t)) * average

# **Step 1**

**Questions 5 and 6: Pricing European Calls and Puts using Heston Model and Monte Carlo**

We assume a correlation value ($\rho$) of -0.3 for the following 4 calculations of European option prices using the two methods. Following which, we repeat the calculations with $\rho = -0.7$.

*European Call - Monte Carlo Method*

In [ ]:
# European Option Pricing - Monte Carlo
# Call
S_ini = 80
r = 0.055
T = 3
t = 0
sigma = 0.35
K = S_ini
option_type = "C"
europrice_c_mc = bs_call_mc(S_ini, K, r, sigma, T, t, 100000)
print(europrice_c_mc)

[[  0.         -26.88494052]
 [  0.           1.71684738]
 [  0.          81.79383354]
 ...
 [  0.          -7.11781881]
 [  0.          72.6218801 ]
 [  0.          -9.4550238 ]]
24.102551385769686


*European Put - Monte Carlo Method*

In [ ]:
# Put
europrice_p_mc = bs_put_mc(S_ini, K, r, sigma, T, t, 100000)
print(europrice_p_mc)

12.117702221858005


*European Call - Heston Model*

In [ ]:
# European Option Pricing - Heston Model
# Call
v0 = 0.032
kappa_v = 1.85
sigma_v = sigma
theta_v = 0.045
rho = - 0.3

M0 = 50
T = 0.25
M = int(M0*T)
Ite = 10000

dt = T/M

rand = random_number_gen(M, Ite)

covariance_matrix = np.zeros((2, 2), dtype=float)
covariance_matrix[0] = [1.0, rho]
covariance_matrix[1] = [rho, 1.0]
cho_matrix = np.linalg.cholesky(covariance_matrix)

V = SDE_vol(v0, kappa_v, theta_v, sigma_v, T, M, Ite, rand, 1, cho_matrix)
S = Heston_paths(S_ini, r, V, 0, cho_matrix)

hestonprice_c_mc = heston_call_mc(S, K, r, T, t)
print(hestonprice_c_mc)

2.706117216082509


*European Put - Heston Model*

In [ ]:
# Put
hestonprice_p_mc = heston_put_mc(S, K, r, T, t)
print(hestonprice_p_mc)

5.525802135411094


We now repeat the calculations with a correlation ($\rho$) value of 0.7:

In [ ]:
# Rho is now - 0.7
# Call
rho = - 0.7

covariance_matrix = np.zeros((2, 2), dtype=float)
covariance_matrix[0] = [1.0, rho]
covariance_matrix[1] = [rho, 1.0]
cho_matrix = np.linalg.cholesky(covariance_matrix)

V = SDE_vol(v0, kappa_v, theta_v, sigma_v, T, M, Ite, rand, 1, cho_matrix)
S = Heston_paths(S_ini, r, V, 0, cho_matrix)

hestonprice_c_mc_2 = heston_call_mc(S, K, r, T, t)
print(hestonprice_c_mc_2)

0.7115829670080303


In [ ]:
# Put
hestonprice_p_mc_2 = heston_put_mc(S, K, r, T, t)
print(hestonprice_p_mc_2)

8.404215630785693


**Question 7: Calculating Delta and Gamma for European Options**

In [ ]:
# BS - European Calls - Calculating the Greeks
option_type = "C"

d1 = (np.log(S_ini / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

if option_type in ["C", "P"]:
  if option_type == "C":
    Opt_Price = S_ini * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    Delta = norm.cdf(d1)
    Gamma = norm.pdf(d1) / (S_ini * sigma * np.sqrt(T))
    Vega = S_ini * np.sqrt(T) * norm.pdf(d1)
    Theta = -(S_ini * sigma * norm.pdf(d1)) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
    Rho = K * T * np.exp(-r * T) * norm.cdf(d2)
  else:
    Opt_Price = K * np.exp(-r * T) * norm.cdf(-d2) - S_ini * norm.cdf(-d1)
    Delta = -norm.cdf(-d1)
    Gamma = norm.pdf(d1) / (S_ini * sigma * np.sqrt(T))
    Vega = norm.pdf(d1) * S_ini * np.sqrt(T)
    Theta = -(S_ini * sigma * norm.pdf(d1)) / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)
    Rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)
else:
  Opt_Price = 'Error: option type incorrect. Choose P for a put option or C for a call option.'

print("Option price = {}".format(Opt_Price))
print("Delta = {}".format(Delta))
print("Gamma = {}".format(Gamma))
print("Vega = {}".format(Vega))
print("Theta = {}".format(Theta))
print("Rho = {}".format(Rho))

Option price = 6.103270146060531
Delta = 0.5659496307005636
Gamma = 0.02810562000354024
Vega = 15.739147201982533
Theta = -13.171901558436923
Rho = 9.793175077496139


In [ ]:
# BS - European Puts - Calculating the Greeks
option_type = "P"

d1 = (np.log(S_ini / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

if option_type in ["C", "P"]:
  if option_type == "C":
    Opt_Price = S_ini * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    Delta = norm.cdf(d1)
    Gamma = norm.pdf(d1) / (S_ini * sigma * np.sqrt(T))
    Vega = S_ini * np.sqrt(T) * norm.pdf(d1)
    Theta = -(S_ini * sigma * norm.pdf(d1)) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
    Rho = K * T * np.exp(-r * T) * norm.cdf(d2)
  else:
    Opt_Price = K * np.exp(-r * T) * norm.cdf(-d2) - S_ini * norm.cdf(-d1)
    Delta = -norm.cdf(-d1)
    Gamma = norm.pdf(d1) / (S_ini * sigma * np.sqrt(T))
    Vega = norm.pdf(d1) * S_ini * np.sqrt(T)
    Theta = -(S_ini * sigma * norm.pdf(d1)) / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)
    Rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)
else:
  Opt_Price = 'Error: option type incorrect. Choose P for a put option or C for a call option.'

print("Option price = {}".format(Opt_Price))
print("Delta = {}".format(Delta))
print("Gamma = {}".format(Gamma))
print("Vega = {}".format(Vega))
print("Theta = {}".format(Theta))
print("Rho = {}".format(Rho))

Option price = 5.010798103424051
Delta = -0.43405036929943647
Gamma = 0.02810562000354024
Vega = 15.739147201982533
Theta = -8.83198752078193
Rho = -9.933706911844743


**Questions 8 and 9: Calculating European Option Prices using the Merton Model**

Central to the Merton model is the usage of a jump parameter, $\lambda$. We first assume $\lambda = 0.75$ in making the calculations for European options, then recalculating after changing the value to $\lambda = 0.25$.

In [ ]:
lamb = 0.75 # Jump parameter
mu = -0.5
delta = 0.22

SM = np.zeros((M + 1, Ite), dtype=float)
SM[0] = S_ini

rj = lamb * (np.exp(mu + 0.5 * delta**2) - 1)

z1 = np.random.standard_normal((M + 1, Ite))
z2 = np.random.standard_normal((M + 1, Ite))
y = np.random.poisson(lamb * dt, (M + 1, Ite))

for t in range(1, M + 1, 1):
  SM[t] = SM[t - 1] * (np.exp((r - rj - 0.5 * sigma**2) * dt + np.sqrt(dt) * sigma * z1[t]) + (np.exp(mu + delta * z2[t]) - 1) * y[t])
  SM[t] = np.maximum(SM[t], 0.00001)

In [ ]:
def merton_call_mc(S, K, r, T, t):
  payoff = np.maximum(0, S[-1, :] - K)
  average = np.mean(payoff)
  return np.exp(-r * (T - t)) * average

In [ ]:
def merton_put_mc(S, K ,r, T, t):
  payoff = np.maximum(0, K - S[-1, :])
  average = np.mean(payoff)
  return np.exp(-r * (T - t)) * average

In [ ]:
mertonprice_c_mc = merton_call_mc(SM, K, r, T, t)
print(mertonprice_c_mc)

16.10344266700229


In [ ]:
mertonprice_p_mc = merton_put_mc(SM, K, r, T, t)
print(mertonprice_p_mc)

13.968380003727932


At this juncture, we switch to $\lambda = 0.25$ and recalculate the European option prices:

In [ ]:
# Lamb is now 0.25
lamb = 0.25 # Jump parameter
mu = -0.5
delta = 0.22

SM = np.zeros((M + 1, Ite), dtype=float)
SM[0] = S_ini

rj = lamb * (np.exp(mu + 0.5 * delta**2) - 1)

z1 = np.random.standard_normal((M + 1, Ite))
z2 = np.random.standard_normal((M + 1, Ite))
y = np.random.poisson(lamb * dt, (M + 1, Ite))

for t in range(1, M + 1, 1):
  SM[t] = SM[t - 1] * (np.exp((r - rj - 0.5 * sigma**2) * dt + np.sqrt(dt) * sigma * z1[t]) + (np.exp(mu + delta * z2[t]) - 1) * y[t])
  SM[t] = np.maximum(SM[t], 0.00001)

In [ ]:
mertonprice_c_mc_2 = merton_call_mc(SM, K, r, T, t)
print(mertonprice_c_mc_2)

13.457817170062812


In [ ]:
mertonprice_p_mc_2 = merton_put_mc(SM, K, r, T, t)
print(mertonprice_p_mc_2)

11.036228917164783


**Question 10: Recalculating Delta and Gamma for European Options**

Having used a model different from the prior two (Heston and Monte Carlo) to calculate European option prices, we recalculate their Delta and Gamma values to check for any changes in value.

In [ ]:
# BS - European Calls - Calculating the Greeks
option_type = "C"

d1 = (np.log(S_ini / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

if option_type in ["C", "P"]:
  if option_type == "C":
    Opt_Price = S_ini * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    Delta = norm.cdf(d1)
    Gamma = norm.pdf(d1) / (S_ini * sigma * np.sqrt(T))
    Vega = S_ini * np.sqrt(T) * norm.pdf(d1)
    Theta = -(S_ini * sigma * norm.pdf(d1)) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
    Rho = K * T * np.exp(-r * T) * norm.cdf(d2)
  else:
    Opt_Price = K * np.exp(-r * T) * norm.cdf(-d2) - S_ini * norm.cdf(-d1)
    Delta = -norm.cdf(-d1)
    Gamma = norm.pdf(d1) / (S_ini * sigma * np.sqrt(T))
    Vega = norm.pdf(d1) * S_ini * np.sqrt(T)
    Theta = -(S_ini * sigma * norm.pdf(d1)) / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)
    Rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)
else:
  Opt_Price = 'Error: option type incorrect. Choose P for a put option or C for a call option.'

print("Option price = {}".format(Opt_Price))
print("Delta = {}".format(Delta))
print("Gamma = {}".format(Gamma))
print("Vega = {}".format(Vega))
print("Theta = {}".format(Theta))
print("Rho = {}".format(Rho))

Option price = 6.103270146060531
Delta = 0.5659496307005636
Gamma = 0.02810562000354024
Vega = 15.739147201982533
Theta = -13.171901558436923
Rho = 9.793175077496139


In [ ]:
# BS - European Puts - Calculating the Greeks
option_type = "P"

d1 = (np.log(S_ini / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

if option_type in ["C", "P"]:
  if option_type == "C":
    Opt_Price = S_ini * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    Delta = norm.cdf(d1)
    Gamma = norm.pdf(d1) / (S_ini * sigma * np.sqrt(T))
    Vega = S_ini * np.sqrt(T) * norm.pdf(d1)
    Theta = -(S_ini * sigma * norm.pdf(d1)) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
    Rho = K * T * np.exp(-r * T) * norm.cdf(d2)
  else:
    Opt_Price = K * np.exp(-r * T) * norm.cdf(-d2) - S_ini * norm.cdf(-d1)
    Delta = -norm.cdf(-d1)
    Gamma = norm.pdf(d1) / (S_ini * sigma * np.sqrt(T))
    Vega = norm.pdf(d1) * S_ini * np.sqrt(T)
    Theta = -(S_ini * sigma * norm.pdf(d1)) / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)
    Rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)
else:
  Opt_Price = 'Error: option type incorrect. Choose P for a put option or C for a call option.'

print("Option price = {}".format(Opt_Price))
print("Delta = {}".format(Delta))
print("Gamma = {}".format(Gamma))
print("Vega = {}".format(Vega))
print("Theta = {}".format(Theta))
print("Rho = {}".format(Rho))

Option price = 5.010798103424051
Delta = -0.43405036929943647
Gamma = 0.02810562000354024
Vega = 15.739147201982533
Theta = -8.83198752078193
Rho = -9.933706911844743


**Question 11: Put-Call Parity**

We recall the equation for put-call parity which is given by

$c_0 + Ke^{-rT} = S_0 + p_0$

Substituting the call and put prices from questions (5), (6), (8) and (9) into the equation, for the prices in question (5) we have

$LHS: 2.68 + 80e^{-0.055(3)} = 70.51$

$RHS: 80 + 5.45 = 85.45$

while for the prices calculated using Monte Carlo simulations we have

$LHS: 24.42 + 80e^{-0.055(3)} = 92.25$

$RHS: 80 + 12.20 = 92.20$

For question (6) we have

$LHS: 0.69 + 80e^{-0.055(3)} = 68.52$

$RHS: 80 + 5.45 = 85.45$

For question (8) we have

$LHS: 16.50 + 80e^{-0.055(3)} = 84.33$

$RHS: 80 + 13.83 = 93.83$

and for question (9) we have

$LHS: 12.97 + 80e^{-0.055(3)} = 80.80$

$RHS: 80 + 11.07 = 91.07$

We thus find that put-call parity holds only for the option prices computed in question (5) using Monte Carlo simulations.

**Question 12: Calculating European Option Prices under 7 Levels of Moneyness (Heston and Merton models)**

The following 7 calculations centre around European options, set at different levels of moneyness ranging from 85% moneyness (deepest OTM) to 115% moneyness (deepest ITM), with the moneyness incrementing upwards by 5% with every stage. Moneyness is defined as the ratio of the strike price to the initial underlying stock price, i.e. the stock price at time $t = 0\ (S_0)$.

In [ ]:
# 7 Option Prices from 0.85-1.15 Moneyness - Heston and Merton
# Calls
# 0.85
K = 0.85*S_ini
hestonprice_c_mc_85 = heston_call_mc(S, K, r, T, t)
print(hestonprice_c_mc_85)
mertonprice_c_mc_85 = merton_call_mc(SM, K, r, T, t)
print(mertonprice_c_mc_85)

12.002612172255684
28.970609607342723


In [ ]:
# 0.9
K = 0.9*S_ini
hestonprice_c_mc_90 = heston_call_mc(S, K, r, T, t)
print(hestonprice_c_mc_90)
mertonprice_c_mc_90 = merton_call_mc(SM, K, r, T, t)
print(mertonprice_c_mc_90)

7.0768913774467395
23.083730467228005


In [ ]:
# 0.95
K = 0.95*S_ini
hestonprice_c_mc_95 = heston_call_mc(S, K, r, T, t)
print(hestonprice_c_mc_95)
mertonprice_c_mc_95 = merton_call_mc(SM, K, r, T, t)
print(mertonprice_c_mc_95)

3.4787262490078663
17.87985324456441


In [ ]:
# 1
K = S_ini
hestonprice_c_mc_100 = heston_call_mc(S, K, r, T, t)
print(hestonprice_c_mc_100)
mertonprice_c_mc_100 = merton_call_mc(SM, K, r, T, t)
print(mertonprice_c_mc_100)

1.3767652698581905
13.457817170062812


In [ ]:
# 1.05
K = 1.05*S_ini
hestonprice_c_mc_105 = heston_call_mc(S, K, r, T, t)
print(hestonprice_c_mc_105)
mertonprice_c_mc_105 = merton_call_mc(SM, K, r, T, t)
print(mertonprice_c_mc_105)

0.48172050776820263
9.871682916778285


In [ ]:
# 1.1
K = 1.1*S_ini
hestonprice_c_mc_110 = heston_call_mc(S, K, r, T, t)
print(hestonprice_c_mc_110)
mertonprice_c_mc_110 = merton_call_mc(SM, K, r, T, t)
print(mertonprice_c_mc_110)

0.16322688874439273
7.039565758275448


In [ ]:
# 1.15
K = 1.15*S_ini
hestonprice_c_mc_115 = heston_call_mc(S, K, r, T, t)
print(hestonprice_c_mc_115)
mertonprice_c_mc_115 = merton_call_mc(SM, K, r, T, t)
print(mertonprice_c_mc_115)

0.05320651417724674
4.866033139499943


In [ ]:
# Puts
# 0.85
K = 0.85*S_ini
hestonprice_p_mc_85 = heston_put_mc(S, K, r, T, t)
print(hestonprice_p_mc_85)
mertonprice_p_mc_85 = merton_put_mc(SM, K, r, T, t)
print(mertonprice_p_mc_85)

3.9858068487252436
3.648569321666556


In [ ]:
# 0.9
K = 0.9*S_ini
hestonprice_p_mc_90 = heston_put_mc(S, K, r, T, t)
print(hestonprice_p_mc_90)
mertonprice_p_mc_90 = merton_put_mc(SM, K, r, T, t)
print(mertonprice_p_mc_90)

6.693570064842348
5.395174192477884


In [ ]:
# 0.95
K = 0.95*S_ini
hestonprice_p_mc_95 = heston_put_mc(S, K, r, T, t)
print(hestonprice_p_mc_95)
mertonprice_p_mc_95 = merton_put_mc(SM, K, r, T, t)
print(mertonprice_p_mc_95)

10.728888947329521
7.824780980740334


In [ ]:
# 1
K = S_ini
hestonprice_p_mc_100 = heston_put_mc(S, K, r, T, t)
print(hestonprice_p_mc_100)
mertonprice_p_mc_100 = merton_put_mc(SM, K, r, T, t)
print(mertonprice_p_mc_100)

16.260411979105893
11.036228917164783


In [ ]:
# 1.05
K = 1.05*S_ini
hestonprice_p_mc_105 = heston_put_mc(S, K, r, T, t)
print(hestonprice_p_mc_105)
mertonprice_p_mc_105 = merton_put_mc(SM, K, r, T, t)
print(mertonprice_p_mc_105)

22.998851227941948
15.083578674806306


In [ ]:
# 1.1
K = 1.1*S_ini
hestonprice_p_mc_110 = heston_put_mc(S, K, r, T, t)
print(hestonprice_p_mc_110)
mertonprice_p_mc_110 = merton_put_mc(SM, K, r, T, t)
print(mertonprice_p_mc_110)

30.313841619844187
19.88494552722951


In [ ]:
# 1.15
K = 1.15*S_ini
hestonprice_p_mc_115 = heston_put_mc(S, K, r, T, t)
print(hestonprice_p_mc_115)
mertonprice_p_mc_115 = merton_put_mc(SM, K, r, T, t)
print(mertonprice_p_mc_115)

37.83730525620309
25.34489691938006


# **Step 2**

**Question 13: Calculating the Price of an American Call using the Heston, Merton and Monte Carlo methods**

We repeat the calculations in questions (5) and (8) for an American call using the Heston model, Monte Carlo simulations and the Merton model. A correlation value of -0.70 was used for the Heston model and a lambda value of 0.25 was used for the Merton model.

In [ ]:
def bs_call_mc_am(S, K, r, sigma, T, t, Ite):
  z = np.random.normal(0, 1, [1, Ite])
  ST = S*np.exp((T - t)*(r - 0.5 * sigma**2) + sigma * np.sqrt(T - t) * z)
  data = np.zeros(np.transpose(ST).shape)
  # data[:, 1] = ST - K # Keeping European Option Price for Comparison
  for i in range(ST.shape[0]):
    for j in range(ST.shape[1]-2, -1, -1):
      if ST[i, j] > ST[i, j+1] - K:
        data[j+1, i] = ST[i, j]
      else:
        data[j+1, i] = ST[i, j+1] - K

  average = np.sum(np.amax(data, axis=1)) / float(Ite)

  return np.exp(-r * (T - t)) * average

In [ ]:
def bs_put_mc_am(S, K, r, sigma, T, t, Ite):
  z = np.random.normal(0, 1, [1, Ite])
  ST = S*np.exp((T - t)*(r - 0.5 * sigma**2) + sigma * np.sqrt(T - t) * z)
  data = np.zeros(np.transpose(ST).shape)
  # data[:, 1] = K - ST # Keeping European Option Price for Comparison
  for i in range(ST.shape[0]):
    for j in range(ST.shape[1]-2, -1, -1):
      if ST[i, j] > K - ST[i, j+1]:
        data[j+1, i] = ST[i, j]
      else:
        data[j+1, i] = K - ST[i, j+1]

  average = np.sum(np.amax(data, axis=1)) / float(Ite)

  return np.exp(-r * (T - t)) * average

In [ ]:
def heston_call_mc_am(S, K, r, T, t):
  data = np.zeros(np.transpose(S).shape)
  for i in range(S.shape[0]):
    for j in range(S.shape[1]-2, -1, -1):
      if S[i, j] > S[i, j+1] - K:
        data[j+1, i] = S[i, j]
      else:
        data[j+1, i] = S[i, j+1] - K
  payoff = np.amax(data, axis=1)
  average = np.mean(payoff)
  return np.exp(-r * (T - t)) * average

In [ ]:
def heston_put_mc_am(S, K, r, T, t):
  data = np.zeros(np.transpose(S).shape)
  for i in range(S.shape[0]):
    for j in range(S.shape[1]-2, -1, -1):
      if S[i, j] > K - S[i, j+1]:
        data[j+1, i] = S[i, j]
      else:
        data[j+1, i] = K - S[i, j+1]
  payoff = np.amax(data, axis=1)
  average = np.mean(payoff)
  return np.exp(-r * (T - t)) * average

In [ ]:
def merton_call_mc_am(S, K, r, T, t):
  data = np.zeros(np.transpose(S).shape)
  for i in range(S.shape[0]):
    for j in range(S.shape[1]-2, -1, -1):
      if S[i, j] > K - S[i, j+1]:
        data[j+1, i] = S[i, j]
      else:
        data[j+1, i] = K - S[i, j+1]
  payoff = np.amax(data, axis=1)
  average = np.mean(payoff)
  return np.exp(-r * (T - t)) * average

In [ ]:
def merton_put_mc_am(S, K, r, T, t):
  data = np.zeros(np.transpose(S).shape)
  for i in range(S.shape[0]):
    for j in range(S.shape[1]-2, -1, -1):
      if S[i, j] > K - S[i, j+1]:
        data[j+1, i] = S[i, j]
      else:
        data[j+1, i] = K - S[i, j+1]
  payoff = np.amax(data, axis=1)
  average = np.mean(payoff)
  return np.exp(-r * (T - t)) * average

In [ ]:
hestonprice_c_mc_am = heston_call_mc_am(S, K, r, T, t)
print(hestonprice_c_mc_am)
mertonprice_c_mc_am = merton_call_mc_am(SM, K, r, T, t)
print(mertonprice_c_mc_am)

156.0823817326294
173.51548469412228


In [ ]:
hestonprice_p_mc_am = heston_put_mc_am(S, K, r, T, t)
print(hestonprice_p_mc_am)
mertonprice_p_mc_am = merton_put_mc_am(SM, K, r, T, t)
print(mertonprice_p_mc_am)

156.0823817326294
173.51548469412228


In [ ]:
bs_c_mc_am = bs_call_mc_am(S_ini, K, r, sigma, 3, 0, Ite)
print(bs_c_mc_am)
bs_p_mc_am = bs_put_mc_am(S_ini, K, r, sigma, 3, 0, Ite)
print(bs_p_mc_am)

85.6670068200965
80.87574350158268


We notice that we obtain significantly higher results using the Heston and Merton models than with Monte Carlo simulations; we believe this is due to the greater spread in possible underlying stock values and hence option prices simulated under the Heston and Merton models as a result of considering correlation and jump intensity values.

**Question 14: Pricing an Up-and-In (UAI) Option using the Heston model**

In [ ]:
# UAI Option - Heston
K = 95
euprice_uai_c = heston_call_mc(S, K, r, T, t)
if euprice_uai_c < 95:
  euprice_uai_c = 0
print(euprice_uai_c)

0


Here, we obtain a call price of \$0 (option expires worthless) as the UAI option did not reach the barrier level over the course of the Heston path simulations. The simple European call retains its value at \$2.68 using the Heston model, as calculated in question (1).

**Question 15: Pricing a Down-and-In (DAI) Put Option using the Merton model**

In [ ]:
# DAI Option - Merton
K = 65
euprice_dai_c = merton_call_mc(SM, K, r, T, t)
if euprice_dai_c > 65:
  euprice_dai_c = 0
print(euprice_dai_c)

33.743053343833814


In this case, we obtain a final call price of \$33.17, which is much greater than (more than double) the European call price of \$16.50.

**Table of Values for Option Prices (Appending Heston and Merton Model Methods)**

In [ ]:
import pandas as pd
type_asset = ["ATM Call", "ATM Put", "ATM Call", "ATM Put"]
exer = ["Eur", "Eur", "Amer", "Amer"]
options_df = pd.DataFrame({
    "Type": type_asset,
    "Exer": exer,
    "GWP1 Method": ["Binomial" for i in range(len(type_asset))],
    "GWP2 Method": ["BS", "BS", "MC", "MC"],
    "GWP3 Method": ["Heston + Merton + MC" for i in range(len(type_asset))],
    "GWP1 Price": [21.68, 7.75, 21.68, 8.89],
    "GWP2 Price": [20.92, 6.99, 101.04, 99.96],
    "GWP3 Price": [str(hestonprice_c_mc) + "(H), " + str(mertonprice_c_mc) + "(M), " + str(europrice_c_mc) + "(MC)",
                   str(hestonprice_p_mc) + "(H), " + str(mertonprice_p_mc) + "(M), " + str(europrice_p_mc) + "(MC)",
                   str(hestonprice_c_mc_am) + "(H), " + str(mertonprice_c_mc_am) + "(M), " + str(bs_c_mc_am) + "(MC)",
                   str(hestonprice_p_mc_am) + "(H), " + str(mertonprice_p_mc_am) + "(M), " + str(bs_p_mc_am) + "(MC)"]
})
options_df.style \
  .format(precision=3, thousands=".", decimal=",") \
  .format_index(str.upper, axis=1) \
  .relabel_index(["5", "5", "8", "8"], axis=0)

,TYPE,EXER,GWP1 METHOD,GWP2 METHOD,GWP3 METHOD,GWP1 PRICE,GWP2 PRICE,GWP3 PRICE
5,ATM Call,Eur,Binomial,BS,Heston + Merton + MC,"21,680","20,920","2.706117216082509(H), 16.10344266700229(M), 24.102551385769686(MC)"
5,ATM Put,Eur,Binomial,BS,Heston + Merton + MC,"7,750","6,990","5.525802135411094(H), 13.968380003727932(M), 12.117702221858005(MC)"
8,ATM Call,Amer,Binomial,MC,Heston + Merton + MC,"21,680","101,040","156.0823817326294(H), 173.51548469412228(M), 85.6670068200965(MC)"
8,ATM Put,Amer,Binomial,MC,Heston + Merton + MC,"8,890","99,960","156.0823817326294(H), 173.51548469412228(M), 80.87574350158268(MC)"


# **References**

WorldQuant University. (2024, July 23). MScFE620: Derivative Pricing (Modules 1-7).